# QuickQuake catalog visualization: Aniakchak example (12 March 2023)

Run `python examples/aniakchak_2023/QQ_driver.py` first (from the QuickQuake folder).

Figures to look at the results of a QuickQuake run, in the order of a paper: where the region
is, what was detected, when, at what depth, how the pipeline moved the events, and what the
signals look like. All the logic lives in `qq_helpers.py`; this notebook only sets paths, loads
the results, calls plotting functions and (optionally) saves the figures in
`DATA_ROOT/images/visualization/`. Edit the **Settings** cell, then run all cells. The analysis
figures (catalog summary, comparison with reference catalogs, magnitude of completeness,
temporal evolution, FI, clusters and planes) are in `catalog_analysis.ipynb`.

**Kernel:** the QuickQuake conda environment (the one `install.sh` creates). In VS Code select it as the kernel; in a terminal run `conda activate <environment>` and then `jupyter lab`.

| Section | Figures |
|---|---|
| 2 | Region and stations: regional location map, eruption history, stations |
| 3 | Seismicity in map view: full map, zoom |
| 4 | Seismicity through time: daily counts, timeline with periods and its legend |
| 5 | Seismicity with depth: map with the profile line, cross section |
| 6 | 3-D views: coloured by date, interactive, animation (GIF) |
| 7 | Location changes GaMMA -> HypoInverse -> HypoDD |
| 8 | Waveforms: move-outs and spectrograms |
| 9 | Quality control of single events: move out and energy-velocity curve |

In [ ]:
from pathlib import Path
import sys
import importlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.io as pio

# This notebook lives in examples/aniakchak_2023/; qq_helpers.py is in the quickquake/ folder of QuickQuake
HELPERS_DIR = (Path.cwd() / ".." / ".." / "quickquake").resolve()
if str(HELPERS_DIR) not in sys.path:
    sys.path.insert(0, str(HELPERS_DIR))

import qq_helpers as h
importlib.reload(h)
print("helpers:", h.__file__)

pio.renderers.default = "plotly_mimetype+notebook_connected"   # interactive figures in VS Code and Jupyter

## 0. Settings
Everything you normally edit is in this cell.

In [ ]:
# ---------------- results of a QuickQuake run ----------------
# Relative paths are relative to this notebook's folder (examples/aniakchak_2023/).
DRIVER = Path("QQ_driver.py").resolve()   # the QQ_driver.py of the run (section 9 uses its QC settings)
DATA_ROOT = Path("data").resolve()        # the "data" folder next to that QQ_driver.py
NAMEBASE = "GAMMA"                        # same as location_namebase in QQ_driver.py
REGION = {"name": "Aniakchak", "lat": 56.9058, "lon": -158.209}     # volcano / region centre (degrees)
RADIUS_KM = 20.0              # keep only events within this distance (km) of the centre; None = keep all
START = "2023-03-12T00:00:00Z"
END   = "2023-03-13T00:00:00Z"

# ---------------- what to draw ----------------
SAVE_FIGURES = True           # save into DATA_ROOT/images/visualization/
DEPTH_MAX_KM = 13.0           # deepest event shown in depth-coloured / 3-D figures (km)
SPANS = []                    # (start, end, label) periods for the timelines; [] = none
PROFILE = dict(start_lon=-158.2729, start_lat=56.87708, end_lon=-157.9812, end_lat=56.87708,
               label_start="C'", label_end="C", swath_km=5.0)   # cross section (swath: km on each side); None = skip


# ---------------- section 2: context ----------------
REGIONAL_EXTENT = (-179, -145, 51, 66)   # (lon_min, lon_max, lat_min, lat_max) of the location map (degrees); Alaska
ERUPTION_HISTORY = None       # {"eruption": [years], "non_eruptive": [years], "xlim": (first, last)}; None = skip

# ---------------- section 6: animation ----------------
MAKE_GIF = False              # the GIF of the 3-D view through time takes a few minutes

# ---------------- paths inside DATA_ROOT (standard QuickQuake layout) ----------------
MERGED = DATA_ROOT / "merged"
CATALOG_CSV   = MERGED / f"output/{NAMEBASE}_hypodd_catalog_noglitch_qcaccepted.csv"   # final QuickQuake catalog
HYPODD_CSV    = MERGED / f"output/{NAMEBASE}_hypodd_catalog.csv"
HYPOINV_CSV   = MERGED / f"output/{NAMEBASE}_hypoinv_good.csv"
GAMMA_CSV     = MERGED / f"input/{NAMEBASE}_catalog_cleaned.csv"
PICKS_CSV     = MERGED / f"input/{NAMEBASE}_picks_cleaned.csv"
STATIONS_JSON = MERGED / f"input/{NAMEBASE}_station_list.json"
WAVEFORM_INDEX = DATA_ROOT / "waveform_index.pkl"


IMAGE_DIR = h.ensure_dir(DATA_ROOT / "images" / "visualization")


def save(fig, name, **kwargs):
    """Save a figure into IMAGE_DIR when SAVE_FIGURES is True."""
    if SAVE_FIGURES and fig is not None:
        out = IMAGE_DIR / name
        if hasattr(fig, "write_html"):
            fig.write_html(out, include_plotlyjs="cdn")
        else:
            fig.savefig(out, bbox_inches="tight", **kwargs)
        print("saved:", out)

## 1. Load the catalog and the stations

In [ ]:
catc = h.CatalogCols(time="time", lat="latitude", lon="longitude", depth_km="depth_km", mag="magnitude")
stc = h.StationCols(station_id="station", lat="latitude", lon="longitude", elev_km="elev_km")

df_catalog = h.load_catalog_csv(CATALOG_CSV, catc=catc)
df_stations = h.load_stations_json(STATIONS_JSON)

# only the stations that were used: at least one pick in an event of the located catalog (the whole
# run, before the radius cut); the others were downloaded but PhaseNet / GaMMA never used them
if PICKS_CSV.exists():
    picks_per_station = h.stations_with_picks(PICKS_CSV, event_ids=pd.read_csv(CATALOG_CSV, usecols=["event_id"])["event_id"])
    used = df_stations["station"].str.split(".").str[:2].str.join(".").isin(picks_per_station.index)
    print("stations without picks, not drawn:", sorted(df_stations.loc[~used, "station"]) or "none")
    df_stations = df_stations[used].reset_index(drop=True)
n_net_sta = df_stations["station"].str.split(".").str[:2].str.join(".").nunique()
print(f"catalog: {len(df_catalog)} events | stations: {n_net_sta} ({len(df_stations)} instruments)")

if RADIUS_KM is not None:
    df_catalog = h.filter_catalog_by_radius(df_catalog, center_lat=REGION["lat"], center_lon=REGION["lon"],
                                            radius_km=RADIUS_KM, lat_col=catc.lat, lon_col=catc.lon)
    print(f"within {RADIUS_KM:g} km of {REGION['name']}: {len(df_catalog)} events "
          f"(farthest {df_catalog['r_km'].max():.2f} km)")

print("time range:", df_catalog[catc.time].min(), "->", df_catalog[catc.time].max())

## 2. Region and stations

In [ ]:
fig, ax = h.plot_regional_location_map(volcano_lon=REGION["lon"], volcano_lat=REGION["lat"],
                                       region_extent=REGIONAL_EXTENT, label=REGION["name"])
save(fig, "location_map_regional.png", dpi=300)
plt.show()

In [ ]:
if ERUPTION_HISTORY is None:
    print("eruption history skipped: set ERUPTION_HISTORY in the Settings cell")
else:
    fig, ax = h.plot_eruption_history(ERUPTION_HISTORY.get("eruption", []), ERUPTION_HISTORY.get("non_eruptive", []),
                                      xlim_years=ERUPTION_HISTORY.get("xlim"))
    save(fig, "eruption_history.png", dpi=600)
    plt.show()

In [ ]:
# the map covers all the stations (and the events); zoom 9 is enough for a region of ~200 km
fig, ax = h.plot_mapview_basemap_stations(catalog=df_catalog, stations=df_stations, start=START, end=END,
                                          relief_zoom=9, step_lat=0.5, figsize=(15, 8), savefig=False)
save(fig, "mapview_stations_only.svg")
plt.show()

## 3. Seismicity in map view

In [ ]:
fig, ax = h.plot_mapview_seismicity(catalog=df_catalog, stations=df_stations, start=START, end=END,
                                    depth_vmin=0.0, depth_vmax=DEPTH_MAX_KM, region_name=REGION["name"],
                                    relief_zoom=12, figsize=(15, 8), savefig=False)
save(fig, "mapview_seismicity.svg")
plt.show()

In [ ]:
# zoom window (degrees): edit these two lines to choose what the zoom shows;
# None = 3 times smaller than the map of all the events, around its centre (zoom_factor)
ZOOM_LON = None   # (west, east)
ZOOM_LAT = None   # (south, north)

fig, ax, meta = h.plot_mapview_seismicity_zoom(catalog=df_catalog, start=START, end=END,
                                               depth_vmin=0.0, depth_vmax=DEPTH_MAX_KM, cmap_name="inferno_r",
                                               relief_zoom=12, step_lon=0.2, step_lat=0.1, figsize=(15, 8),
                                               zoom_factor=3, xlim=ZOOM_LON, ylim=ZOOM_LAT, show=False)
save(fig, "mapview_zoom.svg")
plt.show()
lon_min, lon_max, lat_min, lat_max = meta["bounds"]
in_window = df_catalog[catc.lon].between(lon_min, lon_max) & df_catalog[catc.lat].between(lat_min, lat_max)
print(f"window lon {lon_min:.3f} to {lon_max:.3f}, lat {lat_min:.3f} to {lat_max:.3f}: "
      f"{int(in_window.sum())} of {len(df_catalog)} events")

## 4. Seismicity through time

In [ ]:
fig, ax, counts = h.plot_event_count_time(catalog=df_catalog, time_col=catc.time, start=START, end=END,
                                          freq="D", figsize=(12, 5))
save(fig, "event_count_daily.svg")
plt.show()

In [ ]:
fig, ax, meta = h.plot_timeline_histogram(catalog=df_catalog, start=START, end=END, spans=SPANS,
                                          time_col=catc.time, fi_col="FI", n_bins=90, figsize=(14, 4),
                                          show_legend=False, savefig=False)   # the legend is its own figure (next cell)
save(fig, "timeline_histogram.svg")
plt.show()

In [ ]:
# legend of the timeline as its own figure (FI classes and periods), to place next to it in a paper
fig_leg, _ = h.plot_timeline_legend_only(fi_categories=meta["fi_categories"], fi_colors=meta["fi_colors"],
                                         span_types=meta["types"], span_color_map=meta["color_map"])
save(fig_leg, "timeline_legend.svg")
plt.show()

## 5. Seismicity with depth

In [ ]:
if PROFILE is None:
    print("cross section skipped: set PROFILE in the Settings cell")
else:
    fig, ax = h.plot_mapview_seismicity(catalog=df_catalog, stations=df_stations, start=START, end=END,
                                        depth_vmin=0.0, depth_vmax=DEPTH_MAX_KM, region_name=REGION["name"],
                                        relief_zoom=12, figsize=(15, 8), savefig=False)
    h.add_profile_line(ax=ax, start_lon=PROFILE["start_lon"], start_lat=PROFILE["start_lat"],
                       end_lon=PROFILE["end_lon"], end_lat=PROFILE["end_lat"],
                       start_label=PROFILE["label_start"], end_label=PROFILE["label_end"],
                       line_color="red", line_width=3.0, marker_size=80, fontsize=16)
    save(fig, "mapview_seismicity_with_crosssection.svg")
    plt.show()

    fig, ax, meta = h.plot_cross_section_seismicity(
        catalog=df_catalog, start=START, end=END, catc=catc,
        start_lon=PROFILE["start_lon"], start_lat=PROFILE["start_lat"],
        end_lon=PROFILE["end_lon"], end_lat=PROFILE["end_lat"], swath_km=PROFILE["swath_km"],
        depth_vmin=0.0, depth_vmax=DEPTH_MAX_KM, depth_max_plot=DEPTH_MAX_KM, cmap_name="inferno_r",
        x_label_start=PROFILE["label_start"], x_label_end=PROFILE["label_end"], figsize=(12, 6))
    save(fig, "cross_section.svg")
    plt.show()
    print(meta)

## 6. 3-D views

In [ ]:
# 3-D view coloured by origin date; color_by="month" colours by calendar month (to look for seasonality)
VIEW_ELEV = 10.0    # camera elevation above the horizontal (degrees)
VIEW_AZIM = 265.0   # camera azimuth (degrees, matplotlib view_init)
EXAGG = 1.0         # vertical exaggeration of the topography

fig, ax, meta = h.plot_mapview3d(catalog=df_catalog, start=START, end=END, color_by="time", depth_max=DEPTH_MAX_KM,
                                 dem_res="15s", exagg=EXAGG, view_elev=VIEW_ELEV, view_azim=VIEW_AZIM,
                                 figsize=(12, 9), savefig=False, catc=catc)
save(fig, "mapview3d_time.svg")
plt.show()
print(f"events drawn: {meta['n_events']} ({meta['n_magnitude_le_0']} with magnitude <= 0, at the minimum marker size)")

In [ ]:
fig = h.plot_catalog_3d_topo_seismicity(catalog=df_catalog, catc=catc, mode="time", dem_res="15s",
                                        region_name=REGION["name"], show=False)
save(fig, "catalog_3d_interactive.html")
fig.show()

In [ ]:
if not MAKE_GIF:
    print("animation skipped: set MAKE_GIF = True in the Settings cell (takes a few minutes)")
else:
    out_gif = IMAGE_DIR / "time_evolution.gif"
    h.animate_mapview3d_timecolor(catalog=df_catalog, start=START, end=END, out_gif=str(out_gif),
                                  duration_s=15.0, fps=20, depth_max=DEPTH_MAX_KM, exagg=1.0, size_k=8.0,
                                  marker_min=6.0, topo_stride=1, view_elev=5, view_azim=265, dpi=220, catc=catc)
    print("saved:", out_gif)

## 7. Location changes between pipeline stages
The same events located by GaMMA, HypoInverse and HypoDD; each segment joins two locations of one event.

In [ ]:
# HypoDD stage = the final QC-filtered catalog (as in the paper figures); use HYPODD_CSV for all relocated events
df_loc = h.build_location_evolution_df(GAMMA_CSV, HYPOINV_CSV, CATALOG_CSV, start=START, end=END)
# same events as the rest of the notebook: within RADIUS_KM of the centre at their final (HypoDD) location;
# their GaMMA and HypoInverse locations can be farther away, which is what this figure shows
df_loc = df_loc[df_loc["event_id"].isin(df_catalog["event_id"].astype(int))].reset_index(drop=True)
print(f"{len(df_loc)} events in all three stages | median horizontal shift: "
      f"GaMMA->HypoInverse {df_loc['shift_gamma_hypoinv_km'].median():.2f} km, "
      f"HypoInverse->HypoDD {df_loc['shift_hypoinv_hypodd_km'].median():.2f} km")
fig, axes, drawn = h.plot_location_evolution_map(df_loc, max_events=1200, pad_lon=0.02, pad_lat=0.02,
                                                 x_tick_label_size=15.3, y_tick_label_size=18, legend_fontsize=13)
save(fig, "location_changes_gamma_hypoinv_hypodd.pdf")
plt.show()

## 8. Waveforms: move-outs and spectrograms
Needs the waveform chunks of the run (`DATA_ROOT/<chunk>/waveforms/*.mseed`) and the cleaned picks.

In [ ]:
has_waveforms = PICKS_CSV.exists() and (WAVEFORM_INDEX.exists() or any(DATA_ROOT.glob("*/waveforms/*.mseed")))
if not has_waveforms:
    print("waveform sections skipped: no picks file or no waveforms under DATA_ROOT")
else:
    wf_catalog, wf_picks, wf_stations = h.load_inputs(CATALOG_CSV, PICKS_CSV, STATIONS_JSON)
    # same events as the rest of the notebook (within RADIUS_KM)
    wf_catalog = wf_catalog[wf_catalog["event_id"].isin(df_catalog["event_id"])].sort_values("time").reset_index(drop=True)  # time order, as the QC figures
    print(f"events with waveforms to draw: {len(wf_catalog)}")
    if WAVEFORM_INDEX.exists():
        wf_index = h.WaveformIndex.load(WAVEFORM_INDEX)
    else:
        wf_index = h.WaveformIndex.build([DATA_ROOT], glob_pattern="**/waveforms/*.mseed")
        wf_index.save(WAVEFORM_INDEX)
    stream_cache = h.StreamCache()
    print("indexed mseed files:", len(wf_index.df))

In [ ]:
MOVEOUT_FIRST = 0   # first event drawn: its row in wf_catalog (time order, starts at 0)
MOVEOUT_LAST = 4   # last event drawn (included); edit both before running
MOVEOUT_EVENT_IDS = None   # None = the rows above; a list such as [2, 6, 9] = those event ids;
                           # "qc_plots" = the events with a QC figure, to compare the QC curve with the move out

if has_waveforms:
    if MOVEOUT_EVENT_IDS is None:
        moveout_events = wf_catalog.iloc[MOVEOUT_FIRST:MOVEOUT_LAST + 1]
    else:
        ids = (h.event_ids_from_qc_plots(MERGED / f"output/qc_plots_{NAMEBASE}") if MOVEOUT_EVENT_IDS == "qc_plots"
               else list(MOVEOUT_EVENT_IDS))
        moveout_events = wf_catalog[wf_catalog["event_id"].isin(ids)]
        missing = sorted(set(ids) - set(moveout_events["event_id"]))
        if missing:
            print("not drawn (outside the radius, or not in CATALOG_CSV, e.g. rejected by QC):", missing)
    print(f"drawing {len(moveout_events)} events:", moveout_events["event_id"].tolist())
    for _, row_event in moveout_events.iterrows():
        fig, ax, meta = h.plot_event_moveout_pretty(
            row_event=row_event, picks=wf_picks, stations=wf_stations, cache=stream_cache, wf_index=wf_index,
            pre_s=5, post_s=40, freqmin=1.0, freqmax=15.0)
        eid = row_event["event_id"]
        save(fig, f"moveout_event{eid}_{row_event['time']:%Y%m%dT%H%M%S}.svg")   # same name as the QC figure
        plt.show()
        print(eid, "stations used:", len(meta["stations_used"]))

In [ ]:
N_SPECTROGRAM_EVENTS = 5   # number of events drawn (the first ones of the catalog); edit before running

if has_waveforms:
    for row_idx, row_event in wf_catalog.iloc[:N_SPECTROGRAM_EVENTS].iterrows():
        fig, axes, meta = h.plot_event_spectrograms_vertical_picks_only(
            row_event=row_event, picks=wf_picks, stations=wf_stations, cache=stream_cache, wf_index=wf_index,
            pre_s=30, post_s=90, window_s=2, overlap_frac=0.8, freq_min=0, freq_max=20)
        save(fig, f"spectrograms_event_{row_idx}.png", dpi=200)
        plt.show()

## 9. Quality control of single events
The QC figure of an accepted event and a rejected one. Top: the vertical traces the QC used, each at its
hypocentral distance (x = seconds from origin time); gray is where the QC velocities predict the arrivals,
and red is the QC window at the velocity of the peak of the curve. Bottom: the energy-velocity curve with
its two percentiles (dashed lines); their ratio is the QC value Q, and an event is kept when
Q >= `qc_min_ratio`. The curves are computed again with `QQ_qc.py` and the QC settings of `DRIVER`, so
they are the curves that accepted or rejected the events. Needs the waveforms (first cell of section 8).

In [ ]:
QC_EVENT_IDS = None    # None = the accepted event with the highest Q among those with at least QC_MIN_STATIONS
                       # stations, and the rejected event with the lowest Q; or a list of event ids
QC_MIN_STATIONS = 5

if not has_waveforms:
    print("QC figure skipped: no picks file or no waveforms under DATA_ROOT")
else:
    qc_ids = QC_EVENT_IDS or h.qc_example_event_ids(DATA_ROOT, namebase=NAMEBASE, start=START, end=END,
                                                     center_lat=REGION["lat"], center_lon=REGION["lon"],
                                                     radius_km=RADIUS_KM, min_stations=QC_MIN_STATIONS)
    fig, axes, qc_events = h.plot_qc_events(qc_ids, data_root=DATA_ROOT, driver=DRIVER, namebase=NAMEBASE)
    if fig is not None:
        save(fig, "qc_events.png", dpi=300)
        plt.show()